# Notebook 08 — Variables meteorológicas trimestrales (Open-Meteo)

A partir del dataset target trimestral construido en el notebook 07, este notebook descarga variables meteorológicas históricas con Open-Meteo y las agrega a nivel de trimestre para construir el primer bloque de variables externas del TFG.

El objetivo ya no es modelar un target anual continuo, sino enriquecer el problema de clasificación trimestral con información del contexto meteorológico.

## Objetivo

- Cargar el dataset target trimestral del notebook 07.
- Descargar datos meteorológicos diarios históricos.
- Construir variables agregadas por trimestre.
- Integrarlas con el target trimestral.
- Realizar una inspección descriptiva inicial por clase.
- Guardar el dataset meteorológico trimestral para notebooks posteriores.

## Preparación del entorno

Montaje de Google Drive, importación de librerías y definición de rutas de trabajo.

In [73]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Carga del target trimestral

Se carga el dataset target generado en el notebook 07, que contiene una fila por trimestre y la clase objetivo de actividad (`low`, `medium`, `high`).

In [74]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE_DIR = Path("/content/drive/MyDrive/PIDS4jjj2")
OUT_DIR = BASE_DIR / "outputs" / "llm_activity"
EXTERNAL_DIR = BASE_DIR / "data" / "external"

EXTERNAL_DIR.mkdir(parents=True, exist_ok=True)

TARGET_PATH = OUT_DIR / "valmayor_target_quarter_classification.parquet"

WEATHER_DAILY_PATH = EXTERNAL_DIR / "valmayor_weather_daily_2009_2026.csv"
WEATHER_QUARTER_PATH = EXTERNAL_DIR / "valmayor_weather_by_quarter.csv"
MODEL_WEATHER_PATH = OUT_DIR / "valmayor_target_meteo_quarter.parquet"
MODEL_WEATHER_CSV_PATH = OUT_DIR / "valmayor_target_meteo_quarter.csv"

In [75]:
df_target = pd.read_parquet(TARGET_PATH)

print(df_target.shape)
df_target.head()

(54, 11)


,year,year_quarter,n_videos,mean_activity_mentions,share_low,share_medium,share_high,share_certainty_high,captures_nonnull_mean,target_class,target_class_id
0,2009.0,2009-Q3,2,1.0,1.0,0.0,0.0,0.0,NaN,low,0
1,2009.0,2009-Q4,1,1.0,0.0,1.0,0.0,1.0,1.0,medium,1
2,2010.0,2010-Q1,1,1.0,1.0,0.0,0.0,0.0,NaN,low,0
3,2010.0,2010-Q2,2,1.0,0.0,1.0,0.0,0.5,1.0,medium,1
4,2011.0,2011-Q1,1,1.0,0.0,1.0,0.0,1.0,1.0,medium,1


## Rango temporal del target

Antes de descargar meteorología, se comprueba qué años y trimestres están presentes en el dataset target trimestral.

In [76]:
print("Años disponibles en target:")
display(df_target["year"].value_counts().sort_index())

print("\nTrimestres disponibles en target:")
display(df_target["year_quarter"].value_counts().sort_index())

Años disponibles en target:


,count
year,
2009.0,2
2010.0,2
2011.0,3
2012.0,3
2013.0,2
2014.0,3
2015.0,2
2016.0,3
2017.0,3



Trimestres disponibles en target:


,count
year_quarter,
2009-Q3,1
2009-Q4,1
2010-Q1,1
2010-Q2,1
2011-Q1,1
2011-Q3,1
2011-Q4,1
2012-Q2,1
2012-Q3,1


In [77]:
print("Distribución de clases:")
display(df_target["target_class"].value_counts())

Distribución de clases:


,count
target_class,
low,28
medium,22
high,4


## Descarga de variables meteorológicas diarias (Open-Meteo)

Se descargan variables meteorológicas diarias para la zona de Valmayor. Posteriormente se agregarán a nivel trimestral.

In [78]:
import requests

LAT = 40.54
LON = -4.05

START_DATE = "2009-01-01"
END_DATE   = "2026-04-31"

print("Salida diaria    :", WEATHER_DAILY_PATH)
print("Salida trimestral:", WEATHER_QUARTER_PATH)

Salida diaria    : /content/drive/MyDrive/PIDS4jjj2/data/external/valmayor_weather_daily_2009_2026.csv
Salida trimestral: /content/drive/MyDrive/PIDS4jjj2/data/external/valmayor_weather_by_quarter.csv


In [79]:
url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": LAT,
    "longitude": LON,
    "start_date": START_DATE,
    "end_date": END_DATE,
    "timezone": "Europe/Madrid",
    "daily": ",".join([
        "temperature_2m_mean",
        "temperature_2m_max",
        "temperature_2m_min",
        "precipitation_sum",
        "rain_sum",
        "snowfall_sum",
        "wind_speed_10m_max",
        "shortwave_radiation_sum",
        "et0_fao_evapotranspiration"
    ])
}

r = requests.get(url, params=params, timeout=60)
r.raise_for_status()

data = r.json()

print("Claves respuesta:", data.keys())
print("Claves bloque daily:", data["daily"].keys())

Claves respuesta: dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'daily_units', 'daily'])
Claves bloque daily: dict_keys(['time', 'temperature_2m_mean', 'temperature_2m_max', 'temperature_2m_min', 'precipitation_sum', 'rain_sum', 'snowfall_sum', 'wind_speed_10m_max', 'shortwave_radiation_sum', 'et0_fao_evapotranspiration'])


In [80]:
df_weather_daily = pd.DataFrame(data["daily"])
df_weather_daily["time"] = pd.to_datetime(df_weather_daily["time"])

df_weather_daily["year"] = df_weather_daily["time"].dt.year
df_weather_daily["quarter"] = df_weather_daily["time"].dt.quarter
df_weather_daily["year_quarter"] = (
    df_weather_daily["year"].astype(str) + "-Q" + df_weather_daily["quarter"].astype(str)
)

print(df_weather_daily.shape)
df_weather_daily.head()

(6330, 13)


,time,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,rain_sum,snowfall_sum,wind_speed_10m_max,shortwave_radiation_sum,et0_fao_evapotranspiration,year,quarter,year_quarter
0,2009-01-01,5.1,9.8,2.0,0.7,0.7,0.0,6.7,5.66,0.73,2009,1,2009-Q1
1,2009-01-02,5.3,9.4,1.2,0.0,0.0,0.0,6.2,5.53,0.72,2009,1,2009-Q1
2,2009-01-03,6.3,7.3,5.0,5.4,5.4,0.0,6.9,1.65,0.30,2009,1,2009-Q1
3,2009-01-04,5.0,7.5,0.3,0.0,0.0,0.0,10.0,7.28,0.81,2009,1,2009-Q1
4,2009-01-05,0.9,5.1,-2.7,0.0,0.0,0.0,5.6,5.78,0.65,2009,1,2009-Q1


In [81]:
df_weather_daily = df_weather_daily.rename(columns={
    "time": "date",
    "temperature_2m_mean": "temp_mean",
    "temperature_2m_max": "temp_max",
    "temperature_2m_min": "temp_min",
    "precipitation_sum": "precip_sum",
    "rain_sum": "rain_sum",
    "snowfall_sum": "snowfall_sum",
    "wind_speed_10m_max": "wind_max",
    "shortwave_radiation_sum": "radiation_sum",
    "et0_fao_evapotranspiration": "eto_sum"
})

df_weather_daily.head()

,date,temp_mean,temp_max,temp_min,precip_sum,rain_sum,snowfall_sum,wind_max,radiation_sum,eto_sum,year,quarter,year_quarter
0,2009-01-01,5.1,9.8,2.0,0.7,0.7,0.0,6.7,5.66,0.73,2009,1,2009-Q1
1,2009-01-02,5.3,9.4,1.2,0.0,0.0,0.0,6.2,5.53,0.72,2009,1,2009-Q1
2,2009-01-03,6.3,7.3,5.0,5.4,5.4,0.0,6.9,1.65,0.30,2009,1,2009-Q1
3,2009-01-04,5.0,7.5,0.3,0.0,0.0,0.0,10.0,7.28,0.81,2009,1,2009-Q1
4,2009-01-05,0.9,5.1,-2.7,0.0,0.0,0.0,5.6,5.78,0.65,2009,1,2009-Q1


In [82]:
print(df_weather_daily.dtypes)

date             datetime64[ns]
temp_mean               float64
temp_max                float64
temp_min                float64
precip_sum              float64
rain_sum                float64
snowfall_sum            float64
wind_max                float64
radiation_sum           float64
eto_sum                 float64
year                      int32
quarter                   int32
year_quarter             object
dtype: object


In [83]:
df_weather_daily.to_csv(WEATHER_DAILY_PATH, index=False, encoding="utf-8")
print("Guardado diario en:", WEATHER_DAILY_PATH)

Guardado diario en: /content/drive/MyDrive/PIDS4jjj2/data/external/valmayor_weather_daily_2009_2026.csv


## Agregación meteorológica trimestral

Las variables diarias se resumen por trimestre para hacerlas compatibles con el dataset target del notebook 07.

In [84]:
df_weather_quarter = (
    df_weather_daily
    .groupby(["year", "quarter", "year_quarter"], as_index=False)
    .agg({
        "temp_mean": "mean",
        "temp_max": "mean",
        "temp_min": "mean",
        "precip_sum": "sum",
        "rain_sum": "sum",
        "snowfall_sum": "sum",
        "wind_max": "mean",
        "radiation_sum": "sum",
        "eto_sum": "sum"
    })
    .sort_values(["year", "quarter"])
    .reset_index(drop=True)
)

df_weather_quarter.head(20)

,year,quarter,year_quarter,temp_mean,temp_max,temp_min,precip_sum,rain_sum,snowfall_sum,wind_max,radiation_sum,eto_sum
0,2009,1,2009-Q1,5.631111,10.788889,0.945556,99.3,61.9,27.86,13.473333,1121.00,165.69
1,2009,2,2009-Q2,16.325275,22.480220,9.600000,71.9,71.4,0.35,16.025275,2306.66,460.02
2,2009,3,2009-Q3,23.242391,29.376087,16.431522,35.2,35.2,0.00,13.942391,2235.07,540.72
3,2009,4,2009-Q4,9.527174,14.081522,5.525000,181.0,168.0,9.38,14.471739,869.97,151.84
4,2010,1,2010-Q1,4.318889,8.257778,0.816667,209.6,154.7,38.99,15.778889,913.70,122.37
5,2010,2,2010-Q2,14.508791,19.989011,8.581319,164.4,164.4,0.00,13.484615,2141.95,383.47
6,2010,3,2010-Q3,23.413043,29.611957,16.616304,42.5,42.5,0.00,13.081522,2227.58,525.07
7,2010,4,2010-Q4,7.357609,11.965217,3.385870,187.3,178.5,6.30,13.134783,871.96,137.11
8,2011,1,2011-Q1,5.338889,10.076667,1.298889,172.8,146.7,18.83,13.282222,1000.54,137.13
9,2011,2,2011-Q2,16.658242,22.331868,10.634066,183.2,183.2,0.00,12.889011,2222.57,420.55


In [85]:
df_weather_quarter = df_weather_quarter.rename(columns={
    "temp_mean": "temp_mean_quarter",
    "temp_max": "temp_max_mean_quarter",
    "temp_min": "temp_min_mean_quarter",
    "precip_sum": "precip_sum_quarter",
    "rain_sum": "rain_sum_quarter",
    "snowfall_sum": "snowfall_sum_quarter",
    "wind_max": "wind_max_mean_quarter",
    "radiation_sum": "radiation_sum_quarter",
    "eto_sum": "eto_sum_quarter"
})

df_weather_quarter.head()

,year,quarter,year_quarter,temp_mean_quarter,temp_max_mean_quarter,temp_min_mean_quarter,precip_sum_quarter,rain_sum_quarter,snowfall_sum_quarter,wind_max_mean_quarter,radiation_sum_quarter,eto_sum_quarter
0,2009,1,2009-Q1,5.631111,10.788889,0.945556,99.3,61.9,27.86,13.473333,1121.00,165.69
1,2009,2,2009-Q2,16.325275,22.480220,9.600000,71.9,71.4,0.35,16.025275,2306.66,460.02
2,2009,3,2009-Q3,23.242391,29.376087,16.431522,35.2,35.2,0.00,13.942391,2235.07,540.72
3,2009,4,2009-Q4,9.527174,14.081522,5.525000,181.0,168.0,9.38,14.471739,869.97,151.84
4,2010,1,2010-Q1,4.318889,8.257778,0.816667,209.6,154.7,38.99,15.778889,913.70,122.37


In [86]:
df_weather_quarter.to_csv(WEATHER_QUARTER_PATH, index=False, encoding="utf-8")
print("Guardado trimestral en:", WEATHER_QUARTER_PATH)

Guardado trimestral en: /content/drive/MyDrive/PIDS4jjj2/data/external/valmayor_weather_by_quarter.csv


## Merge con el target trimestral

Se integran las variables meteorológicas trimestrales con el target de clasificación para construir el primer dataset de modelado enriquecido con variables externas.

In [87]:
df_model_weather = (
    df_target
    .merge(df_weather_quarter, on=["year", "year_quarter"], how="left")
    .sort_values(["year", "year_quarter"])
    .reset_index(drop=True)
)

print(df_model_weather.shape)
df_model_weather.head(20)

(54, 21)


,year,year_quarter,n_videos,mean_activity_mentions,share_low,share_medium,share_high,share_certainty_high,captures_nonnull_mean,target_class,...,quarter,temp_mean_quarter,temp_max_mean_quarter,temp_min_mean_quarter,precip_sum_quarter,rain_sum_quarter,snowfall_sum_quarter,wind_max_mean_quarter,radiation_sum_quarter,eto_sum_quarter
0,2009.0,2009-Q3,2,1.000000,1.000000,0.000000,0.000000,0.000000,NaN,low,...,3,23.242391,29.376087,16.431522,35.2,35.2,0.00,13.942391,2235.07,540.72
1,2009.0,2009-Q4,1,1.000000,0.000000,1.000000,0.000000,1.000000,1.0,medium,...,4,9.527174,14.081522,5.525000,181.0,168.0,9.38,14.471739,869.97,151.84
2,2010.0,2010-Q1,1,1.000000,1.000000,0.000000,0.000000,0.000000,NaN,low,...,1,4.318889,8.257778,0.816667,209.6,154.7,38.99,15.778889,913.70,122.37
3,2010.0,2010-Q2,2,1.000000,0.000000,1.000000,0.000000,0.500000,1.0,medium,...,2,14.508791,19.989011,8.581319,164.4,164.4,0.00,13.484615,2141.95,383.47
4,2011.0,2011-Q1,1,1.000000,0.000000,1.000000,0.000000,1.000000,1.0,medium,...,1,5.338889,10.076667,1.298889,172.8,146.7,18.83,13.282222,1000.54,137.13
5,2011.0,2011-Q3,3,1.000000,0.000000,1.000000,0.000000,0.666667,1.0,medium,...,3,22.677174,28.991304,15.803261,16.8,16.8,0.00,13.904348,2267.14,533.76
6,2011.0,2011-Q4,2,1.000000,0.500000,0.500000,0.000000,0.500000,1.0,low,...,4,9.427174,14.466304,5.168478,155.1,155.1,0.00,11.976087,917.63,153.48
7,2012.0,2012-Q2,1,1.000000,1.000000,0.000000,0.000000,0.000000,NaN,low,...,2,16.069231,21.776923,9.843956,94.8,94.2,0.42,16.680220,2187.36,440.52
8,2012.0,2012-Q3,2,1.000000,1.000000,0.000000,0.000000,0.000000,NaN,low,...,3,23.165217,29.498913,16.117391,73.5,73.5,0.00,15.317391,2255.36,552.54
9,2012.0,2012-Q4,1,1.000000,0.000000,1.000000,0.000000,0.000000,1.0,medium,...,4,8.297826,12.431522,4.772826,180.4,180.4,0.00,12.640217,809.79,126.30


## Validación tras el merge

Se comprueba si el cruce ha sido correcto y si quedan nulos en las variables meteorológicas agregadas.

In [88]:
weather_cols = [c for c in df_model_weather.columns if c.endswith("_quarter") or c in ["year", "quarter"]]

print("Nulos por columna meteorológica:")
display(df_model_weather[weather_cols].isna().sum())

Nulos por columna meteorológica:


,0
year,0
year_quarter,0
quarter,0
temp_mean_quarter,0
temp_max_mean_quarter,0
temp_min_mean_quarter,0
precip_sum_quarter,0
rain_sum_quarter,0
snowfall_sum_quarter,0
wind_max_mean_quarter,0


In [89]:
summary_by_class = (
    df_model_weather
    .groupby("target_class")
    .agg(
        n_trimesters=("year_quarter", "count"),
        n_videos_mean=("n_videos", "mean"),
        temp_mean_quarter=("temp_mean_quarter", "mean"),
        temp_max_mean_quarter=("temp_max_mean_quarter", "mean"),
        precip_sum_quarter=("precip_sum_quarter", "mean"),
        rain_sum_quarter=("rain_sum_quarter", "mean"),
        wind_max_mean_quarter=("wind_max_mean_quarter", "mean"),
        radiation_sum_quarter=("radiation_sum_quarter", "mean")
    )
    .round(3)
)

summary_by_class

,n_trimesters,n_videos_mean,temp_mean_quarter,temp_max_mean_quarter,precip_sum_quarter,rain_sum_quarter,wind_max_mean_quarter,radiation_sum_quarter
target_class,,,,,,,,
high,4,2.250,9.961,14.960,188.325,183.125,15.060,1246.383
low,28,3.179,16.570,21.992,128.371,122.968,16.043,1766.390
medium,22,2.318,12.881,17.997,152.450,147.359,15.304,1380.870


## Conclusión

Este notebook adapta el bloque meteorológico del pipeline al nuevo enfoque del TFG. En lugar de construir variables anuales para un problema de regresión, se generan variables meteorológicas trimestrales compatibles con el dataset target de clasificación del notebook 07.

El resultado es un dataset ya enriquecido con el primer bloque de variables externas, listo para ser complementado en los notebooks siguientes con hidrología y presión atmosférica.

In [90]:
df_model_weather.to_parquet(MODEL_WEATHER_PATH, index=False)
df_model_weather.to_csv(MODEL_WEATHER_CSV_PATH, index=False, encoding="utf-8")

print("Guardado parquet model weather:", MODEL_WEATHER_PATH)
print("Guardado csv model weather:", MODEL_WEATHER_CSV_PATH)

Guardado parquet model weather: /content/drive/MyDrive/PIDS4jjj2/outputs/llm_activity/valmayor_target_meteo_quarter.parquet
Guardado csv model weather: /content/drive/MyDrive/PIDS4jjj2/outputs/llm_activity/valmayor_target_meteo_quarter.csv


In [91]:
df_check = pd.read_parquet(MODEL_WEATHER_PATH)

print("Shape recargado:", df_check.shape)
df_check.head()

Shape recargado: (54, 21)


,year,year_quarter,n_videos,mean_activity_mentions,share_low,share_medium,share_high,share_certainty_high,captures_nonnull_mean,target_class,...,quarter,temp_mean_quarter,temp_max_mean_quarter,temp_min_mean_quarter,precip_sum_quarter,rain_sum_quarter,snowfall_sum_quarter,wind_max_mean_quarter,radiation_sum_quarter,eto_sum_quarter
0,2009.0,2009-Q3,2,1.0,1.0,0.0,0.0,0.0,NaN,low,...,3,23.242391,29.376087,16.431522,35.2,35.2,0.00,13.942391,2235.07,540.72
1,2009.0,2009-Q4,1,1.0,0.0,1.0,0.0,1.0,1.0,medium,...,4,9.527174,14.081522,5.525000,181.0,168.0,9.38,14.471739,869.97,151.84
2,2010.0,2010-Q1,1,1.0,1.0,0.0,0.0,0.0,NaN,low,...,1,4.318889,8.257778,0.816667,209.6,154.7,38.99,15.778889,913.70,122.37
3,2010.0,2010-Q2,2,1.0,0.0,1.0,0.0,0.5,1.0,medium,...,2,14.508791,19.989011,8.581319,164.4,164.4,0.00,13.484615,2141.95,383.47
4,2011.0,2011-Q1,1,1.0,0.0,1.0,0.0,1.0,1.0,medium,...,1,5.338889,10.076667,1.298889,172.8,146.7,18.83,13.282222,1000.54,137.13


In [92]:
print("Distribución de clases tras integrar meteo:")
display(df_check["target_class"].value_counts())

Distribución de clases tras integrar meteo:


,count
target_class,
low,28
medium,22
high,4
